# Orpheus-3B Sunbird Luganda — Inference

Standalone inference notebook for the finetuned Luganda voice at [`sunbird/orpheus-3b-tts-salt-lug-0001`](https://huggingface.co/sunbird/orpheus-3b-tts-salt-lug-0001). No training, no data prep — just load the model and synthesize speech from arbitrary Luganda text.

**Use cases:**
- Audition the model on sentences it has never seen.
- A/B against the held-out `test` split for quality eyeballing.
- Lift the `synthesize()` function into a FastAPI / vLLM deployment with no notebook-specific code to remove.

**Companion notebook**: `Orpheus_3B_Sunbird_Luganda.ipynb` trains and pushes the model this notebook consumes.

## #0 Setup & secrets

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install snac torchcodec "datasets>=3.4.1,<4.0.0"
!pip install soundfile

In [ ]:
import os
from getpass import getpass

# HF token only — needed to pull the model from sunbird/orpheus-3b-tts-salt-lug-0001.
# Skip the prompt if the model is already on disk locally and HF_TOKEN is set in env.
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

In [ ]:
import locale
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
from datasets import load_dataset, Audio
from snac import SNAC
from IPython.display import display, Audio as AudioDisplay

locale.getpreferredencoding = lambda: "UTF-8"

INFERENCE_OUTPUT_DIR = Path("orpheus-3B/inference_outputs")
INFERENCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## #1 Load the finetuned model + SNAC decoder

Set `MODEL_ID` to either an HF repo or a local checkpoint path (`outputs/orpheus_finetune_16bit/` is what the training notebook produces). `SPEAKER_ID` must match the multi-speaker tag the model was trained with — `salt_lug_0001` for the default Luganda voice. `LOAD_IN_4BIT=True` cuts VRAM in half for deployment on smaller GPUs at a small quality cost.

In [ ]:
'''
from huggingface_hub import HfApi, hf_hub_download
import json, pathlib

REPO = MODEL_ID   # e.g. "patrickcmd/orpheus-3b-tts-multilingual"

local_cfg = pathlib.Path(hf_hub_download(REPO, "config.json", token=os.environ["HF_TOKEN"]))
cfg = json.loads(local_cfg.read_text())
print("current config.vocab_size =", cfg["vocab_size"])

cfg["vocab_size"] = 156939
out = pathlib.Path("config_fixed.json")
out.write_text(json.dumps(cfg, indent=2))

HfApi().upload_file(
    path_or_fileobj=str(out),
    path_in_repo="config.json",
    repo_id=REPO,
    commit_message="fix(config): vocab_size 156940 -> 156939 to match embed weights",
    token=os.environ["HF_TOKEN"],
)
print("hub config.json patched — re-run #1 load cell.")

'''

In [ ]:
'''
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download

snap = Path(snapshot_download(repo_id=MODEL_ID, token=os.environ["HF_TOKEN"]))
patched = Path("orpheus-3B/outputs/orpheus_patched")
patched.mkdir(parents=True, exist_ok=True)

for name in os.listdir(snap):
    src, dst = snap / name, patched / name
    if dst.exists() or dst.is_symlink():
        continue
    if name == "config.json":
        cfg = json.loads(src.read_text())
        if cfg.get("vocab_size") != 156939:
            print(f"patch: vocab_size {cfg.get('vocab_size')} -> 156939")
        cfg["vocab_size"] = 156939
        dst.write_text(json.dumps(cfg, indent=2))
    else:
        dst.symlink_to(src.resolve())

# Then load from the local patched dir instead of the hub repo:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(patched),
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    token = os.environ.get("HF_TOKEN"),
)
FastLanguageModel.for_inference(model)

'''

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "sunbird/orpheus-3b-tts-salt-lug-0001"
# Or load locally:
# MODEL_ID = "orpheus-3B/outputs/orpheus_finetune_16bit"

SPEAKER_ID    = "salt_lug_0001"
LOAD_IN_4BIT  = False           # True → ~6 GB VRAM, slight quality drop
MAX_SEQ_LEN   = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,                   # auto bf16 / fp16
    load_in_4bit = LOAD_IN_4BIT,
    token = os.environ.get("HF_TOKEN"),
)
FastLanguageModel.for_inference(model)   # 2x faster generate
print(f"Loaded LM: {MODEL_ID}")

In [ ]:
# SNAC decoder runs on CPU — frees GPU for the LM, decoding is fast enough.
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cpu")
print("Loaded SNAC 24 kHz decoder (on CPU)")

## #2 Token constants + helpers

Same special-token layout as the training notebook. The audio codebook spans `[128266, 128266 + 7·4096) = [128266, 156938)`; anything outside that range in the model's output is treated as garbage and filtered out before SNAC decoding.

In [ ]:
# Special tokens (must match #2.6 of the training notebook)
START_OF_TEXT   = 128000
END_OF_TEXT     = 128009
START_OF_SPEECH = 128257
END_OF_SPEECH   = 128258
START_OF_HUMAN  = 128259
END_OF_HUMAN    = 128260
PAD_TOKEN       = 128263
AUDIO_TOKEN_LO  = 128266
AUDIO_TOKEN_HI  = 128266 + 7 * 4096   # exclusive (156938)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = PAD_TOKEN


def _build_input_ids(prompt: str, speaker_id: str) -> torch.Tensor:
    """Wrap `f"{speaker_id}: {prompt}"` in [SOH] ... [EOT, EOH]."""
    tagged = f"{speaker_id}: {prompt}"
    text_ids = tokenizer(tagged, return_tensors="pt").input_ids
    soh = torch.tensor([[START_OF_HUMAN]], dtype=torch.int64)
    end = torch.tensor([[END_OF_TEXT, END_OF_HUMAN]], dtype=torch.int64)
    return torch.cat([soh, text_ids, end], dim=1)


def _redistribute_codes(code_list: list[int]) -> torch.Tensor:
    """Inverse of the SNAC encoding. Returns a (1, 1, n_samples) waveform."""
    layer_1, layer_2, layer_3 = [], [], []
    for i in range(len(code_list) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - 2*4096)
        layer_3.append(code_list[7*i + 3] - 3*4096)
        layer_2.append(code_list[7*i + 4] - 4*4096)
        layer_3.append(code_list[7*i + 5] - 5*4096)
        layer_3.append(code_list[7*i + 6] - 6*4096)
    if not layer_1:
        return torch.zeros(1, 1, 12000)   # ~0.5s silence fallback
    def clamp(vals):
        return [max(0, min(4095, v)) for v in vals]
    codes = [torch.tensor(clamp(layer_1)).unsqueeze(0),
             torch.tensor(clamp(layer_2)).unsqueeze(0),
             torch.tensor(clamp(layer_3)).unsqueeze(0)]
    return snac_model.decode(codes)

## #3 `synthesize()` — the deployable entry point

Single function: `text in → 24 kHz mono float32 numpy array out`. No notebook globals beyond `model`, `tokenizer`, `snac_model` (loaded in #1) and the helpers from #2 — so to deploy you just lift #1 + #2 + #3 into a Python module.

**Sampling defaults** (`temperature=0.6`, `top_p=0.95`, `repetition_penalty=1.1`) come from the upstream Orpheus reference notebook. `max_new_tokens=1200` gives ~9–10 seconds of audio at SNAC's 7-codes/frame rate; raise it if your prompts run longer.

In [ ]:
def synthesize(
    text: str,
    speaker_id: str = SPEAKER_ID,
    *,
    max_new_tokens: int = 1200,
    temperature: float = 0.6,
    top_p: float = 0.95,
    repetition_penalty: float = 1.1,
    seed: int | None = None,
) -> np.ndarray:
    """Synthesize speech for `text`.

    Returns a 1-D float32 numpy array at 24 kHz mono.
    Pass an int `seed` for reproducible output.
    """
    if seed is not None:
        torch.manual_seed(seed)

    input_ids      = _build_input_ids(text, speaker_id).to("cuda")
    attention_mask = torch.ones_like(input_ids)

    generated = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        max_new_tokens = max_new_tokens,
        do_sample = True,
        temperature = temperature,
        top_p = top_p,
        repetition_penalty = repetition_penalty,
        eos_token_id = END_OF_SPEECH,
        use_cache = True,
    )

    # Crop on last SOS — discards the echoed prompt and any pre-speech tokens.
    sos_indices = (generated == START_OF_SPEECH).nonzero(as_tuple=True)
    if len(sos_indices[1]) > 0:
        last = sos_indices[1][-1].item()
        cropped = generated[:, last + 1:]
    else:
        print("WARNING: no start-of-speech token in output; output may be silence.")
        cropped = generated

    # Filter to audio-codebook range only (strips EOS + any garbage tokens).
    row = cropped[0]
    audio_only = row[(row >= AUDIO_TOKEN_LO) & (row < AUDIO_TOKEN_HI)]
    n = (audio_only.size(0) // 7) * 7
    code_list = [t.item() - AUDIO_TOKEN_LO for t in audio_only[:n]]

    waveform = _redistribute_codes(code_list)
    return waveform.detach().squeeze().to("cpu").numpy().astype(np.float32)

## #4 Free-text inference — try your own Luganda sentence

Edit the `prompt` variable below and re-run the cell to hear the model on any new text. Output is saved to `orpheus-3B/inference_outputs/free_text.wav` and displayed inline.

In [ ]:
prompt = "Mwattu, oli otya?"   # ← put any Luganda sentence here

wav = synthesize(prompt, seed=42)
out_path = INFERENCE_OUTPUT_DIR / "free_text.wav"
sf.write(str(out_path), wav, 24000)
print(f"{prompt}")
print(f"  -> {out_path}  ({len(wav)/24000:.2f}s)")
display(AudioDisplay(wav, rate=24000))

## #5 Held-out test split — A/B vs ground truth

The `test` split was never used during training. For each of the first 5 utterances, this cell synthesizes the model's version, saves it next to the ground-truth recording, and displays both inline so you can A/B them.

In [ ]:
ds_test = load_dataset("Sunbird/tts", "lug", split="test")
ds_test = ds_test.filter(lambda r: r["speaker_id"] == SPEAKER_ID)
ds_test = ds_test.cast_column("audio", Audio(sampling_rate=24000))
print(f"{len(ds_test)} test rows for {SPEAKER_ID}")

n_compare = min(5, len(ds_test))
for i in range(n_compare):
    row = ds_test[i]
    text = row["text"]
    print(f"\n[{i}] {text[:120]}")

    wav = synthesize(text, seed=42 + i)
    synth_path = INFERENCE_OUTPUT_DIR / f"test_{i:02d}_synth.wav"
    truth_path = INFERENCE_OUTPUT_DIR / f"test_{i:02d}_groundtruth.wav"
    sf.write(str(synth_path), wav, 24000)
    sf.write(str(truth_path), row["audio"]["array"], 24000)
    print(f"    synth        -> {synth_path}  ({len(wav)/24000:.2f}s)")
    print(f"    ground truth -> {truth_path}")
    print("    [synth]")
    display(AudioDisplay(wav, rate=24000))
    print("    [ground truth]")
    display(AudioDisplay(row["audio"]["array"], rate=24000))

## #6 Deployment notes

The `synthesize()` function in #3 has no notebook-specific dependencies — it only uses `model`, `tokenizer`, `snac_model` (loaded in #1) and the constants/helpers in #2. To turn this into a service:

1. Copy #1 + #2 + #3 into a Python module (e.g. `orpheus_inference.py`). The `model = ...`, `tokenizer = ...`, `snac_model = ...` lines become module-level globals loaded at import time.
2. Wrap `synthesize()` in your transport of choice — see the FastAPI snippet below for a minimal HTTP endpoint.
3. Deploy on a GPU instance (≥8 GB VRAM with `LOAD_IN_4BIT=True`, ≥16 GB without). One worker per GPU; `synthesize()` is single-stream because `model.generate` holds the GPU.

**Latency rough numbers** (RTX 4090, ~5 word sentence):
- generate: ~2–4 s wall clock
- SNAC decode (CPU): ~50–150 ms
- Total: ~3 s for ~5 s of audio (real-time factor ~0.6)

For higher throughput, batch inference (multiple prompts in one `generate` call with left-padding — see #4.2 of the training notebook for the pattern) or serve with **vLLM** which supports Llama-architecture models out of the box.

In [ ]:
# FastAPI deployment snippet — DO NOT RUN inside this notebook.
# Save as `serve.py`, then `uvicorn serve:app --host 0.0.0.0 --port 8000`.
FASTAPI_SNIPPET = '''
import io
import soundfile as sf
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel

from orpheus_inference import synthesize   # #1 + #2 + #3 in a .py module

app = FastAPI()

class TTSRequest(BaseModel):
    text: str
    speaker_id: str = "salt_lug_0001"
    seed: int | None = None

@app.post("/tts", responses={200: {"content": {"audio/wav": {}}}})
def tts(req: TTSRequest):
    wav = synthesize(req.text, req.speaker_id, seed=req.seed)
    buf = io.BytesIO()
    sf.write(buf, wav, 24000, format="WAV", subtype="PCM_16")
    return Response(content=buf.getvalue(), media_type="audio/wav")
'''
print(FASTAPI_SNIPPET)